In [29]:
using Pkg
Pkg.activate(".")
using Distributed
using CSV, DataFrames, BSON, Random

  Activating project at `c:\Users\mikul\Desktop\Persistence of Shocks\tvPersistence.jl - Cleaned`


In [30]:
rmprocs(workers())
num_workers = 4            # ← set to number of CPU cores you want to use
num_replicates = 4        # ← set as desired
addprocs(num_workers)

┌ Warning: rmprocs: process 1 not removed
└ @ Distributed C:\Users\mikul\AppData\Local\Programs\Julia-1.11.4\share\julia\stdlib\v1.11\Distributed\src\cluster.jl:1049


4-element Vector{Int64}:
 14
 15
 16
 17

In [31]:
# (these were read from example_config.txt in the original script)
@everywhere begin
    data_file                    = "data/median_RV.csv"
    data_column                  = "x1"
    missingstring                = "NA"

    ar_order                     = 1
    in_sample_window_size        = 1000
    forecast_horizon             = 1
    forecast_length              = 300
    random_seed                  = 1234

    smoothing_bandwidth          = 0.05
    cutoff_start_index           = 100

    benchmark_method             = "TVAR"
    comparison_method            = "tvEWD"

    tvp_kernel_width             = 0.4
    kernel_type                  = "Epanechnikov"
    max_ar_order                 = 1
    jmax_scale                   = 5
    ar_lag_for_trend             = 1
    tvp_constant_kernel_width    = 0.1
    irf_kernel_width             = 0.2
    forecast_kernel_width        = 0.5
    smoothing_kernel             = "one-sided"
    kernel_type_tvEWD            = "Epanechnikov"
    kernel_type_tvHAR            = "Epanechnikov"
    kernel_type_tvAR             = "Epanechnikov"

    alpha_level                  = 0.05
end

In [32]:
const SED_PATH = abspath("src/SED_Thresholds/SEDThresholds.jl")

"c:\\Users\\mikul\\Desktop\\Persistence of Shocks\\tvPersistence.jl - Cleaned\\src\\SED_Thresholds\\SEDThresholds.jl"

In [33]:
@everywhere include($SED_PATH)        # <— absolute path shipped to workers
@everywhere using .SEDThresholds

In [34]:
# load
df = CSV.File(data_file, missingstring=[missingstring], header=true) |> DataFrame;

# turn column name into a Symbol, drop missings & scale
col_sym = Symbol(data_column);
series  = Float64.(df[.!ismissing.(df[!, col_sym]), col_sym]);

In [35]:
sed_vals = pmap(1:num_replicates) do i
    # re-seed for reproducibility
    Random.seed!(random_seed + i)

    calculate_bootstrap_threshold_parallel(
        i, series,
        ar_order, in_sample_window_size, forecast_horizon,
        smoothing_bandwidth,
        Symbol(benchmark_method), Symbol(comparison_method);
        fcast_len                  = forecast_length,
        tvp_kernel_width           = tvp_kernel_width,
        kernel_type_tvEWD          = kernel_type_tvEWD,
        kernel_type_tvHAR          = kernel_type_tvHAR,
        kernel_type_tvAR           = kernel_type_tvAR,
        smoothing_kernel           = smoothing_kernel,
        max_ar_order               = max_ar_order,
        jmax_scale                 = jmax_scale,
        ar_lag_for_trend           = ar_lag_for_trend,
        tvp_constant_kernel_width  = tvp_constant_kernel_width,
        irf_kernel_width           = irf_kernel_width,
        forecast_kernel_width      = forecast_kernel_width
    )
end;

# remove working processes
rmprocs(workers())

      From worker 16:	[ Info: Performing boostrap simulation number 3
      From worker 14:	[ Info: Performing boostrap simulation number 1
      From worker 15:	[ Info: Performing boostrap simulation number 2
      From worker 17:	[ Info: Performing boostrap simulation number 4


RemoteException: On worker 14:
DimensionMismatch: arrays could not be broadcast to a common size: a has axes Base.OneTo(2258) and b has axes Base.OneTo(300)
Stacktrace:
  [1] _bcs1
    @ .\broadcast.jl:528 [inlined]
  [2] _bcs
    @ .\broadcast.jl:522 [inlined]
  [3] broadcast_shape
    @ .\broadcast.jl:516 [inlined]
  [4] combine_axes
    @ .\broadcast.jl:497 [inlined]
  [5] instantiate
    @ .\broadcast.jl:307 [inlined]
  [6] materialize
    @ .\broadcast.jl:872 [inlined]
  [7] SED_smooth_one
    @ c:\Users\mikul\Desktop\Persistence of Shocks\tvPersistence.jl - Cleaned\src\SED_Thresholds\sed_smoother.jl:29
  [8] SED_smooth_one
    @ c:\Users\mikul\Desktop\Persistence of Shocks\tvPersistence.jl - Cleaned\src\SED_Thresholds\sed_smoother.jl:29 [inlined]
  [9] #calculate_bootstrap_threshold_parallel#15
    @ c:\Users\mikul\Desktop\Persistence of Shocks\tvPersistence.jl - Cleaned\src\SED_Thresholds\bootstrap_thresholds.jl:346
 [10] #21
    @ c:\Users\mikul\Desktop\Persistence of Shocks\tv

In [26]:
sed_vals

4-element Vector{Vector{Float64}}:
 [NaN, -1.670970836459674e-7, -1.0235691986206673e-6, 7.071789375688135e-7, 7.964926988913865e-7, 4.523486821805473e-7, -6.466942694116257e-7, -7.303198395884149e-6, -7.331818913778263e-6, -5.160733251696559e-6  …  -4.403271984475447e-7, -8.88883827503489e-8, 1.1917949354695511e-8, 2.8161865351844154e-7, 3.3643540065088167e-7, 2.8078641552597896e-7, 3.651400777515297e-7, 4.331158485931501e-7, 5.279055326409877e-7, 8.376287216030313e-7]
 [NaN, -1.4590608646253415e-5, -3.3015180368447406e-5, 3.7304668617715658e-6, -7.422525712566413e-7, -6.072331832901066e-6, -9.697024129988306e-6, -8.949461752853666e-6, -7.053617604144404e-6, -1.8397155535828378e-6  …  -7.851014058120455e-6, -5.827468099695721e-6, -4.307876590666331e-6, -3.238036587006386e-6, -1.8012705573486928e-6, -1.9950192552461847e-6, -6.642829652617286e-7, 1.929045487539477e-7, 9.429594074382116e-7, 1.6902298651414534e-6]
 [NaN, -5.026913462114658e-5, -4.1751447743126186e-5, -5.534337771468007e-5

In [27]:
# 1) Count NaNs in each inner vector
nan_counts_per_series = map(v -> count(isnan, v), sed_vals)

# 2) Total number of NaNs across all series
total_nans = sum(nan_counts_per_series)

4

In [28]:
thr = SEDThresholds.compute_global_threshold(sed_vals, cutoff_start_index, alpha_level)
println("SED threshold: ", thr)

SED threshold: 2.936337273608098e-6


In [ ]:
# Save the SED values into BSON file
BSON.@save "sed_thresholds.bson" sed_vals thr